In [1]:
# ===== SETUP: cai dat, tai du lieu, khoi tao Spark =====
import os, time
!pip install -q pyspark
if not os.path.exists("ml-25m/ratings.csv"):
    !wget -q https://files.grouplens.org/datasets/movielens/ml-25m.zip && unzip -q -o ml-25m.zip && rm -f ml-25m.zip

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, FloatType, LongType
spark = (SparkSession.builder.master("local[*]")
         .config("spark.driver.memory", "20g")
         .config("spark.sql.shuffle.partitions", "16")
         .getOrCreate())

# ===== P1.1: CSV -> Parquet =====
schema = StructType([
    StructField("userId", IntegerType()),
    StructField("movieId", IntegerType()),
    StructField("rating", FloatType()),
    StructField("timestamp", LongType()),
])

t0 = time.time()
spark.read.csv("ml-25m/ratings.csv", header=True, schema=schema) \
     .write.mode("overwrite").parquet("data/ratings.parquet")
print(f"Ghi ratings Parquet: {time.time() - t0:.1f}s")

movies = spark.read.option("escape", '"').csv("ml-25m/movies.csv", header=True, inferSchema=True)
movies.write.mode("overwrite").parquet("data/movies.parquet")
print("So phim:", movies.count())

!du -sh ml-25m/ratings.csv data/ratings.parquet

t0 = time.time()
n = spark.read.parquet("data/ratings.parquet").count()
print(f"Parquet: {n} dong, {time.time() - t0:.1f}s")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/23 02:31:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Ghi ratings Parquet: 36.3s


So phim: 62423
647M	ml-25m/ratings.csv
155M	data/ratings.parquet
Parquet: 25000095 dong, 0.9s


In [2]:
# ===== P1.3 + P1.4: tap con theo user + train/test 80/20 =====
import pandas as pd
os.makedirs("results", exist_ok=True)
ratings = spark.read.parquet("data/ratings.parquet")

# Xao tron user 1 lan (seed 42), cong don so rating -> tap long nhau 1M < 5M < 10M < 25M
uc = ratings.groupBy("userId").count().toPandas()
uc = uc.sample(frac=1, random_state=42).reset_index(drop=True)
uc["cum"] = uc["count"].cumsum()

subsets = {}
for name, target in {"1m": 1_000_000, "5m": 5_000_000, "10m": 10_000_000}.items():
    users = spark.createDataFrame(uc.loc[uc["cum"] <= target, ["userId"]])
    subsets[name] = ratings.join(users, "userId")
subsets["25m"] = ratings

rows = []
for name, df in subsets.items():
    t0 = time.time()
    df = df.cache()
    train, test = df.randomSplit([0.8, 0.2], seed=42)
    train.write.mode("overwrite").parquet(f"data/{name}/train")
    test.write.mode("overwrite").parquet(f"data/{name}/test")
    n = df.count()
    u = df.select("userId").distinct().count()
    m = df.select("movieId").distinct().count()
    rows.append({"subset": name, "ratings": n, "users": u, "movies": m,
                 "sparsity": round(1 - n / (u * m), 5),
                 "train": spark.read.parquet(f"data/{name}/train").count(),
                 "test": spark.read.parquet(f"data/{name}/test").count(),
                 "seconds": round(time.time() - t0, 1)})
    df.unpersist()
    print(rows[-1])

stats = pd.DataFrame(rows)
stats.to_csv("results/subsets.csv", index=False)
stats

{'subset': '1m', 'ratings': 999935, 'users': 6580, 'movies': 19991, 'sparsity': 0.9924, 'train': 800018, 'test': 199917, 'seconds': 17.8}


{'subset': '5m', 'ratings': 4999764, 'users': 32665, 'movies': 35730, 'sparsity': 0.99572, 'train': 4000402, 'test': 999362, 'seconds': 16.7}


{'subset': '10m', 'ratings': 9999663, 'users': 65080, 'movies': 45417, 'sparsity': 0.99662, 'train': 7999819, 'test': 1999844, 'seconds': 20.5}


{'subset': '25m', 'ratings': 25000095, 'users': 162541, 'movies': 59047, 'sparsity': 0.9974, 'train': 19999248, 'test': 5000847, 'seconds': 34.2}


,subset,ratings,users,movies,sparsity,train,test,seconds
0,1m,999935,6580,19991,0.99240,800018,199917,17.8
1,5m,4999764,32665,35730,0.99572,4000402,999362,16.7
2,10m,9999663,65080,45417,0.99662,7999819,1999844,20.5
3,25m,25000095,162541,59047,0.99740,19999248,5000847,34.2


In [3]:
!rm -rf ml-25m
!du -sh data results

419M	data
8.0K	results
